In [13]:

import cvxpy as cp
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Flatten, Dense, Softmax
import matplotlib.pyplot as plt
import time
import os
os.environ['CUDA_VISIBLE_DEVICE'] = '0'
from IPython.display import clear_output

In [14]:
B_max = 2
rmax = 1

In [15]:

# def check_constraint(state, action, precision):
#     B1, B2, rem_1, rem_2, h1, h2 = states[0], states[1], states[2], states[3], states[4], states[5]
#     P1, P2, rho1, rho2 = actions[0], actions[1], actions[2], actions[3]
#     flag = np.zeros(7)

#     m1 = np.log(1 + h1* P1)
#     m2 = np.log(1 + h2*P2)
#     m3 = np.log(1 + h1*P1 + h2*P2)

#     if ( P1 < -precision) or (P2 < -precision):
#         flag[0] = 1
#     elif (rho1 < -precision or rho2 < -precision):
#         flag[1] =  1
#     elif (P1 - B1) > precision or (P2 - B2) > precision:
#         flag[2] = 1
#     elif (rho1 - rem_1) > precision or (rho2 - rem_2) > precision:
#         flag[3] = 1
#     elif (P1 > 0 and rho1 < 0) or (P2 > 0 and rho2 < 0 ):
#         flag[4] = 1
#     elif (rho1 - m1) > precision or (rho2 - m2) > precision:
#         flag[5] = 1
#     elif ( (rho1 + rho2) - m3) > precision:
#         flag[6]= 1
#     else:
#         flag = np.zeros(7)

#     return flag


In [16]:

def arrival_E_pkt( pkt_prob, e_prob, weight_prob):
    h_bad = 0.5
    h1_prob = [h_bad, (1-h_bad)]
    h_val = [0.1, 1] 
    weight_val = [1, 2]
    pkt1 = np.random.choice([0, 1], p=[ 1- pkt_prob, pkt_prob])
    E1 = np.random.choice([0, 1], p=[ 1- e_prob, e_prob])
    h1_val= np.random.choice([0.2, 1], p = h1_prob)
    pkt2 = np.random.choice([0, 1], p=[1-pkt_prob,pkt_prob])
    E2 = np.random.choice([0, 1], p=[ 1- e_prob, e_prob])
    h2_val = np.random.choice([0.2, 1], p = h1_prob)
    wt1 = np.random.choice(weight_val, p=[weight_prob , (1-weight_prob)])
    wt2 = np.random.choice(weight_val, p=[weight_prob , (1-weight_prob)])
    arrivals = [E1, E2, pkt1, pkt2, h1_val, h2_val, wt1, wt2]
    return arrivals

In [17]:
P1 = 1
P2 = 1

print(np.log(1+P1)+np.log(1+P2))
print(np.log(1+P1+P1)) 

1.3862943611198906
1.0986122886681098


In [18]:
def state_next(state_slot, actions, arrivals):
    B1, B2 , rem1, rem2, h1, h2, wt_start1, wt_start2 = state_slot
    P1, P2, rho1, rho2 = actions
    
    p1 = max(0, min(P1, B1))
    p2 = max(0, min(P2, B2))
    rho_1 = max(0, min(rho1, rem1))
    rho_2 = max(0, min(rho2, rem2))
    if (rho1 > np.log(1+ h1*p1)*np.log(2) ):
        rho1 = np.log(1 + h1*p1)*np.log(2)
    if (rho2 > np.log(1 + h2*p2)*np.log(2)):
        rho2 = np.log(1 + h2*p2)*np.log(2)

    rem_end_1 = rem1 - rho_1
    rem_end_2 = rem2 - rho_2

    distortion = ( wt_start1 * np.exp(-(rmax -rem_end_1 )) ) +  ( wt_start2 * np.exp(-(rmax - rem_end_2 )) )

    E1, E2, pkt1, pkt2, h1_val, h2_val, wt1, wt2  = arrivals[0], arrivals[1], arrivals[2], arrivals[3], arrivals[4], arrivals[5],arrivals[6], arrivals[7]
    B1_ch = min(B1-p1+E1, B_max)
    B2_ch = min(B2-p2+E2, B_max)

    if pkt1 == 0:
        rem_bits_1_ch = rem1 - rho_1
    else:
        rem_bits_1_ch = rmax
        wt_start1 = wt1

    if pkt2 == 0:
        rem_bits_2_ch = rem2 - rho_2
    else:
        rem_bits_2_ch = rmax
        wt_start2 = wt2
    
    next_state = [B1_ch, B2_ch, rem_bits_1_ch, rem_bits_2_ch, h1_val, h2_val, wt_start1, wt_start2]
    
    return next_state, distortion

In [19]:
def train_data(no_epochs, Time_slot, pkt_prob, e_prob, weight_prob, send_end):
    name_states = 'Training Data/W/train_states_W_'+str(weight_prob) + '.npy'
    name_actions = 'Training Data/W/train_action_W_'+str(weight_prob) + '.npy'
    xs = np.load(name_states)
    ys = np.load(name_actions)
    model = tf.keras.Sequential([
        Flatten(input_shape=(8, 1)),
        Dense(500, activation='relu'),
        Dense(500, activation='relu'),
        Dense(4, activation = 'linear')])

    model.compile(optimizer='adam', loss='mean_squared_error')
    history  = model.fit(xs, ys, epochs=no_epochs, batch_size = 2000, verbose = 0)

    # model_name  = 'Training Data/models/my_model_pkt_'+ str(pkt_prob)
   
    # model.save(model_name)

    print('model saved', str(weight_prob))

    state_slot = np.array([ 0, 0, 0, 0, 0.1, 0.1, 1, 1])
    M = 2
    tot_dist = 0
    import time
    t_cal =  0
    # count_constrint_check = 0
    for t in range(1, Time_slot+1):
        
        state = state_slot.reshape(1, 8)
        t1 = time.time()
        act = model.predict(state, verbose = 0)
        print(time.time()-t1)
        t_cal+= time.time() - t1
        actions = act[0]

        # if np.sum(check_constraint ( state[0], act[0], 0.1 )) > 0:
        #     count_constrint_check = count_constrint_check + 1
        arrivals = arrival_E_pkt(pkt_prob, e_prob, weight_prob)

        next_state, dist = state_next(state_slot, actions,  arrivals)

        tot_dist += dist

        state_slot = np.array(next_state)
        if t %1000 == 0:
            print('time step', weight_prob, t, tot_dist/((t)*M))

    # const_evolve_data.append( count_constrint_check / Time_slot)

    k1 = tot_dist/(Time_slot*M)
    print('time evolve done', weight_prob, k1)
    print('avg time', t_cal/20)


    # send_end.send(k1)
train_data(500, 20, 0.5, 0.5, 0.5, 0)

model saved 0.5
0.04725384712219238
0.025561094284057617
0.025287628173828125
0.024752378463745117
0.025026798248291016
0.02458667755126953
0.024616718292236328
0.026180267333984375
0.028295040130615234
0.031028270721435547
0.027581453323364258
0.02544093132019043
0.026168346405029297
0.024494647979736328
0.024670839309692383
0.025593280792236328
0.026969432830810547
0.024857759475708008
0.02808666229248047
0.025989532470703125
time evolve done 0.5 0.9393958332558631
avg time 0.027151072025299074


In [8]:
lambda_arr = [1.0]
no_epochs = 500
Time_slot = 20000
for weight_prob in lambda_arr:
    train_data(no_epochs, Time_slot, 0.5, 0.5, weight_prob, 0)

2024-11-22 12:19:09.014482: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2024-11-22 12:19:09.014662: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)
2024-11-22 12:19:09.014701: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (sysad-HP-Elite-Tower-600-G9-Desktop-PC): /proc/driver/nvidia/version does not exist
2024-11-22 12:19:09.016304: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


model saved 1.0
time step 1.0 1000 0.5983387032865553
time step 1.0 2000 0.6005191853248526
time step 1.0 3000 0.6007939424498911
time step 1.0 4000 0.6028952081327598
time step 1.0 5000 0.6027640370363893
time step 1.0 6000 0.6020467710018942
time step 1.0 7000 0.6022816918384188
time step 1.0 8000 0.60268596561677
time step 1.0 9000 0.6041382148747907
time step 1.0 10000 0.6046794359428688
time step 1.0 11000 0.604159848576146
time step 1.0 12000 0.6042345944202617
time step 1.0 13000 0.6044866640859168
time step 1.0 14000 0.6049926290427594
time step 1.0 15000 0.6046792136371725
time step 1.0 16000 0.6042602880600365
time step 1.0 17000 0.6045445666936903
time step 1.0 18000 0.6046157800272443
time step 1.0 19000 0.6050392953435486
time step 1.0 20000 0.6047049243333532
time evolve done 1.0 0.6047049243333532


In [9]:
# from time import sleep
# from random import random
# from multiprocessing import Process
# import multiprocessing

# no_epochs = 500
# Time_slot = 20000

# # lambda_arr = np.arange(0.1, 1.1, 0.1)
# lambda_arr = [0.1, 0.2]
# jobs = []
# pipe_list = []
# for pkt_prob in lambda_arr:

#     recv_end, send_end = multiprocessing.Pipe(False)

#     p = Process(target=train_data, args=(no_epochs, Time_slot,  pkt_prob, 0.5, 0.5, send_end))

#     jobs.append(p)
#     pipe_list.append(recv_end)

# for process in jobs:
#     process.start()
# for process in jobs:
#     process.join()

# opt_val_NN = [x.recv() for x in pipe_list]
# print(opt_val_NN)

# # save_name = 'NN_pkt_'+str(no_epochs)+ '_' +str(Time_slot)+'.npy'
# # np.save(save_name, opt_val_NN)

In [10]:

# fig, ax = plt.subplots(figsize=(8, 6))
# plt.plot(lambda_arr, opt_val_NN, color= 'r', label = 'NN')
# plt.show()


In [11]:
# opt_val_NN